# Part 1. Equation of a Slime

How many late days are you using for this assignment?

0

In [225]:
# Imports section
import numpy as np
import pandas as pd
import sklearn 
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

## 1. Loading the dataset

In [226]:
# Using pandas load the dataset
# Output the first 15 rows of the data
# Display a summary of the table information (number of datapoints, etc.)

df = pd.read_csv("science_data_large.csv")
print(df.head(15))
print("Data Point Count:", len(df))
print("Column Names:", df.columns.to_list())




    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04
Data Point Count: 1000
Column Names: ['Temperature °C', 'Mols KCL', 'Size nm^3']


## 2. Splitting the dataset

In [227]:
# Take the pandas dataset and split it into our features (X) and label (y)

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
# For grading consistency use random_state=42 

x_train, x_test, y_train, y_test = train_test_split(df.drop('Size nm^3', axis=1), df['Size nm^3'], train_size=0.9, random_state=42)

## 3. Perform a Linear Regression

In [228]:
# Use sklearn to train a model on the training set

# Create a sample datapoint and predict the output of that sample with the trained model

# Report the score for that model using the default score function property of the SKLearn model, in your own words (markdown, not code) explain what the score means

# Extract the coefficients and intercept from the model and write an equation for your h(x) using LaTeX

linReg = LinearRegression()
fit_linReg = linReg.fit(x_train, y_train)

# create sample datapoint, test model on it
samplePt = pd.DataFrame({'Temperature °C': [500], 'Mols KCL': [450]})
predValue = fit_linReg.predict(samplePt)
print("Sample Datapoint Prediction:", round(predValue[0], 5))

# report score on test data
linRegScore = fit_linReg.score(x_test, y_test)
print("Test Data Score:", round(linRegScore, 5), "\n")

# find coefficients for regression line
print("Y-Intercept:", round(fit_linReg.intercept_, 5))
print("Temperature Coefficient:", round(fit_linReg.coef_[0], 5))
print("KCL Coefficient:", round(fit_linReg.coef_[1], 5))

Sample Datapoint Prediction: 488394.50702
Test Data Score: 0.85525 

Y-Intercept: -409391.47958
Temperature Coefficient: 866.14641
KCL Coefficient: 1032.69507


Write the linear equation of a slime: (example equation: $E = mc^2$)    
h(x) = -409391.47958 + 866.14641(temperature) + 1032.69507(mols)

Report on score and explain meaning:    
The score for the linear regression model represents the coefficient of determination for the test data. In the context of this problem, this coefficient represents how well our model explains the variation in slime size given the slime temperature and potassium chloride. The higher the score, the better our linear regression model does at predicting different slime sizes in our test data.

## 4. Use Cross Validation

In [229]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
# For grading consistency use n_splits=5 and random_state=42

# Report on their finding and their significance

cv = KFold(n_splits=5, random_state=42, shuffle=True)
scores = cross_val_score(linReg, df.drop('Size nm^3', axis=1), df['Size nm^3'], cv=cv, scoring='r2')

for i in range(5):
    print(f"Split {i+1} Score:", round(scores[i], 5))
print("Mean Score:", round(scores.mean(), 5))


Split 1 Score: 0.86152
Split 2 Score: 0.82742
Split 3 Score: 0.87195
Split 4 Score: 0.88166
Split 5 Score: 0.85609
Mean Score: 0.85973


Write findings here:    

We found that the average score using a cross-validation method is 0.85973. This is a more reliable estimation of the linear regression model's performance against unseen data because it considers multiple possible test sets instead of just 1. This means this score is a more accurate representation on how a linear regression model will generalize to unknown data.

## 5. Using Polynomial Regression

In [230]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
# Perform k-fold cross validation (as above)

# Report on the metrics and output the resultant equation as you did in Part 3.

#######################################
# get polynomial features
poly = PolynomialFeatures(degree=2, include_bias=False)
polyFeatureData = poly.fit_transform(df.drop('Size nm^3', axis=1))
polyFeatureDF = pd.DataFrame(polyFeatureData, columns=['Temperature °C', 'Mols KCL', 'Temperature^2', 'Temperature * KCL', 'KCL^2']) # column names omitted since unnecessary for cross validation

# get training and test datasets for linear regression model
x_train, x_test, y_train, y_test = train_test_split(polyFeatureDF, df['Size nm^3'], train_size=0.9, random_state=42)

# run linear regression on training and test data
linReg = LinearRegression()
fit_linReg = linReg.fit(x_train, y_train)

# evaluate linear regression performance, based on our chosen train/test dataset split
linRegScore = fit_linReg.score(x_test, y_test)
print("Test Data Score:", round(linRegScore, 5), "\n")

# identify coefficients of linear regression model
print("Y-Intercept:", round(fit_linReg.intercept_, 5))
print("Temperature Coefficient:", round(fit_linReg.coef_[0], 5))
print("KCL Coefficient:", round(fit_linReg.coef_[1], 5))
print("Temperature^2 Coefficient", round(fit_linReg.coef_[2], 5))
print("Temperature * KCL Coefficient", round(fit_linReg.coef_[3], 5))
print("KCL^2 Coefficient", round(fit_linReg.coef_[4], 5), "\n")

# run cross validation using linReg model on augmented dataset
cv = KFold(n_splits=5, random_state=42, shuffle=True)
scores = cross_val_score(linReg, polyFeatureDF, df['Size nm^3'], cv=cv, scoring='r2')

for i in range(5):
    print(f"Split {i+1} Score:", round(scores[i], 5))
print("Mean Score:", round(scores.mean(), 5))

Test Data Score: 1.0 

Y-Intercept: 2e-05
Temperature Coefficient: 12.0
KCL Coefficient: -0.0
Temperature^2 Coefficient 0.0
Temperature * KCL Coefficient 2.0
KCL^2 Coefficient 0.02857 

Split 1 Score: 1.0
Split 2 Score: 1.0
Split 3 Score: 1.0
Split 4 Score: 1.0
Split 5 Score: 1.0
Mean Score: 1.0


Write the polynomial equation of a slime: (example equation: $E = mc^2$) 
h(x) = 0.00002 + 12(temperature) + 2(temperature * kcl) + 0.02857(kcl^2)

Report on the score and interpret:  
The score of the linear regression model represents the coefficient of determination for the test data. In context, it represents how well the linReg model explains the variation in slime size given its temperature, kcl, temperature^2, temperature * kcl, and kcl^2. The linear regression model has a coefficient of determination of 1, which means that our model predicts the slime sizes of the test data set with perfect accuracy. Additionally, after performing a cross validation of the linear regression model, we see that all 5 of the linear regression models predict slime size with perfect accuracy as well. This makes it less likely that our chosen linReg model is giving us biased performance estimates, and we can expect our model to generalize well to unknown data.

# Part 2. Chronic Kidney Disease Prediction via Classification

Create code and markdown cells as needed to perform classification and report on your results

In [231]:
# Load the dataset
df = pd.read_csv("ckd_feature_subset.csv")
print(df.head())

# Establish random seed of 42
np.random.seed(42)
x = df.drop('Target_ckd', axis=1)
y = df['Target_ckd']

# Establish our four models
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'SVM': SVC(random_state=42),
    'kNN': KNeighborsClassifier(),
    'Neural Network': MLPClassifier(random_state=42)
}

        age        bp      wbcc  appet_poor  appet_good      rbcc  Target_ckd
0  0.688312  0.333333  0.000000           1           0  0.000000           1
1  0.545455  0.333333  0.128319           1           0  0.305085           1
2  0.714286  0.500000  0.238938           1           0  0.186441           1
3  0.688312  0.333333  0.283186           0           1  0.338983           1
4  0.441558  0.333333  0.221239           1           0  0.220339           1


In [232]:
# Evaluate our models using cross-validation
cv = KFold(n_splits=5, random_state=42, shuffle=True)
results = {}

for name, model in models.items():
    scores = cross_val_score(model, x, y, cv=cv)
    results[name] = {
        'Accuracy Mean': round(np.mean(scores), 5),
        'Accuracy Standard Deviation': round(np.std(scores), 5)
    }

/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_

## Results and Conclusion for Classification Experiments

In [233]:
# convert classification results into dataframe, output them
resultsDF = pd.DataFrame.from_dict(results, orient='index')
print(resultsDF)

                     Accuracy Mean  Accuracy Standard Deviation
Logistic Regression        0.85656                      0.06627
SVM                        0.92817                      0.04760
kNN                        0.92796                      0.05244
Neural Network             0.87634                      0.06870


From the above classification experiments, we have estimates for the classification accuracy for 4 different classifiers with random_state=42. Using cross validation to evaluate each model, we see that the classifier that generated models with the highest mean accuracy across the 5 folds was the kNN classifier with k=5(default). Additionally, the classifier that generated models with the lowest standard deviation across the 5 folds was also kNN with k=5. These performance metrics suggest that the kNN classifier with k=5 might be the best at predicting whether or not an unknown patient has breast cancer. 

## Results and Conclusion for Neural Network Experiments

In [234]:
# define parameters for 3 separate neural network configurations settings
configs = [
    {
        'hidden_layer_sizes': (25,),
        'activation': 'tanh', 
        'solver': 'lbfgs',
        'max_iter': 1000
    },
    {
        'hidden_layer_sizes': (50, 25),
        'activation': 'relu',
        'solver': 'adam',
        'max_iter': 1000
    },
    {
        'hidden_layer_sizes': (100, 50, 25),
        'activation': 'logistic',
        'solver': 'sgd',
        'max_iter': 1000
    }
]

# evaluate performance of each neural network
results = {}
for i, config in enumerate(configs, 1):
    model = MLPClassifier(random_state=42, **config)
    scores = cross_val_score(model, x, y, cv=cv)
    results[f'Configuration {i}'] = {
        'Accuracy Mean': np.mean(scores),
        'Accuracy Standard Deviation': np.std(scores)
    }

# create and output dataframe of neural network config dictionary
nnDf = pd.DataFrame.from_dict(results, orient='index')
print(nnDf)

                 Accuracy Mean  Accuracy Standard Deviation
Configuration 1       0.973763                     0.013127
Configuration 2       0.967527                     0.035340
Configuration 3       0.726237                     0.049226


From the 3 different classification settings tested on, I generated 3 different neural network that, when fit to our dataset, generate models for future patients. I gathered the results of running cross validation on these 3 models after being fit to our original dataset. The configuration with the highest mean accuracy across each of the 5 cross validation folds was generated by configuration 1, with accuracy mean of 0.973763. I notice that this accuracy is both about as accurate as configuration 2, and also significantly more accurate than the previously used models. Additionally, the model with the lowest accuracy score standard deviation was also generated by configuration 1. In both accuracy mean and accuracy std, configuration 3 generated the worst of the 3 cross validation model sets.

Interpreting these results requires us to examine the chosen parameters. As both the total hidden neurons and total hidden neuron layers increased, the accuracy mean decreased and accuracy std increased. This suggests that having a more simplified neural network may lend to better results for data of such a small size(n=154). 

Also, the models generated using a logistic sigmoid function had a much worse accuracy mean and accuracy std than models generated using other functions. However, this may be because this configuration also had the most hidden layers and hidden neurons of any models, and I know that the logistic sigmoid function is ideal in this scenario. 

Lastly, I varied the solver for weight optimization. In my experiment, the stochastic gradient descent solver 'sgd' had the worst results. This may be due to
the fact that stochastic gradient descent is not as effective when dealing with small datasets, or it may be simply because other parameter choices in C3 were poor.

These 3 classification settings illustrate that the hidden layers of a neural network are very influential in determining the accuracy of the model the network generates. In this case, since the number of data points was small, my experiments suggested that a more simplified model with fewer layers and overall neurons resulted in a more accurate model. This may be because with too many neurons, the model overfits to the training data, which would explain why it doesn't generalize well to the test data. If we were dealing with a dataset with 100,000 patient records, for instance, I imagine that classification settings with more neurons and layers would score even better on the training, essentially an opposite result of our scenario with not many data points.

______________________________________________________________________________________________________________________________________
______________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________
